In [52]:
from dotenv import load_dotenv

load_dotenv()

True

In [53]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

In [54]:
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부에 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4.7},
    ),
    Document(
        page_content="모공을 커버하는 매트한 파운데이션입니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 4.5},
    ),
    Document(
        page_content="식물성 성분의 저자극 클렌징 오일입니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 4.8},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림입니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4.6},
    ),
    Document(
        page_content="선명한 발색의 롱래스팅 립스틱입니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4.4},
    ),
    Document(
        page_content="SPF50+/PA++++ 톤업 선크림입니다.",
        metadata={"year": 2024, "category": "선케어", "user_rating": 4.9},
    ),
]

In [55]:
vectorstore = Chroma.from_documents(
    documents = docs,
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")
)

In [73]:
metadata_field_info = [
    AttributeInfo(
        name = "category",
        description = "화장품 카테고리. 반드시 '스킨케어', '메이크업', '클렌징', '선케어' 중 하나인 문자열",
        type = "string",
    ),
    AttributeInfo(
        name="year",
        description="제품 출시 연도. 반드시 숫자로 표현하며 예: 2023",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="사용자 평점. 반드시 숫자(float)로 표현하며 예: 4.5",
        type="float",
    ),
]

In [74]:
from langchain_community.query_constructors.chroma import ChromaTranslator

llm = ChatOpenAI(model="gpt-4o", temperature=0)

retriever = SelfQueryRetriever.from_llm(
    llm = llm,
    vectorstore = vectorstore,
    document_contents = "화장품 상품의 설명",
    metadata_field_info = metadata_field_info,
)

In [75]:
query = "2023년에 출시된 스킨케어 제품 중 평점이 '4.5' 이상인 제품을 찾아줘."

results = retriever.invoke(query)

print(len(results))

for doc in results:
    print(doc.page_content)
    print(doc.metadata)

3
비타민 C 함유 브라이트닝 크림입니다.
{'year': 2023, 'user_rating': 4.6, 'category': '스킨케어'}
비타민 C 함유 브라이트닝 크림입니다.
{'category': '스킨케어', 'year': 2023, 'user_rating': 4.6}
비타민 C 함유 브라이트닝 크림입니다.
{'year': 2023, 'user_rating': 4.6, 'category': '스킨케어'}


In [35]:
for field in metadata_field_info:
    print(field.name, "|", field.description, "|", field.type)

category | 화장품 카테고리. 반드시 '스킨케어', '메이크업', '클렌징', '선케어' 중 하나 | string
year | 출시 연도 | integer
user_rating | 사용자 평점 1에서 5까지 | float


In [22]:
structured_query = retriever.query_constructor.invoke({
    "query": "2023년에 출시된 스킨케어 제품 중 평점이 4.5 이상인 제품을 찾아줘."
})

print(structured_query)
print("검색 필터:", structured_query.filter)

query=' ' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2023), Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='user_rating', value=4.5)]) limit=None
검색 필터: operator=<Operator.AND: 'and'> arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2023), Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='user_rating', value=4.5)]
